<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/microscopy-Core-ISMMS/ImageAnalysisCourse/blob/2026-workshop/notebooks/00_data_sources.ipynb)

*Click the badge to open this notebook in Google Colab.*

# Notebook T0 — Data Sources Reference

**Purpose.** This is a reference notebook, not a tutorial. Use it when you want to run any of the workshop notebooks on **data other than the defaults**. Each section below is a self-contained, copy-paste block for one way to load images.

**How the workshop notebooks consume data.** Every T1 lab notebook (NB01, NB02, NB03a/b, NB06, NB07, NB09, NB12, NB13, NB14, NB16) starts with a "Choose data source" decision block. By default that block fetches from the MABC-hosted samples; if MABC is unavailable, it falls through to a canonical published dataset (BBBC, GigaDB, etc.); if that fails, it falls through to the synthetic generator.

The fourth option in that block is **"My own data → see T0"** — that's what brings you here. The blocks below show you how to load your own data into a Python list called `real_imgs`. Once you've run the appropriate block, **switch back to the lab notebook** and re-run from the cell after the decision block. The lab notebook will pick up `real_imgs` from memory.

> ⚠️  Variable name contract: the lab notebooks expect `real_imgs` to be a list (or array) of NumPy arrays. Each notebook also wants specific working variable names bound from `real_imgs` — e.g. NB01 expects `img_easy`, `img_hard`. See the **Variable name reference** at the end of this notebook for the per-notebook contract.

---

## What the workshop uses

Every workshop notebook is designed to run end-to-end **without downloading any large external dataset**. The lab notebooks use one of three sources, layered as a fallback chain:

### scikit-image built-in datasets

Bundled with the `scikit-image` package — no download required after `pip install scikit-image`.

| Function | What it returns | Used in |
|---|---|---|
| `skimage.data.cell()` | A 2D fluorescence single-cell image | NB01 (real-data section), canonical fallback |
| `skimage.data.human_mitosis()` | A 2D fluorescence nuclei image | NB01, NB05 sample exports |
| `skimage.data.cells3d()` | A 3D confocal stack — `(z=60, c=2, y=256, x=256)` with DAPI + membrane | NB01 (mid-Z slice for nuclei seg), NB05, NB06 fallback |

Real microscopy images, openly redistributable, Cellpose-compatible for the easy-case demos.

### Synthetic generators

Each lab that needs a controlled-truth example generates its own synthetic image inline, so the truth count and topology are fully known. No external data, no download.

- **NB01** — `make_easy_image()` (12 round well-separated cells) and `make_hard_image()` (18 irregular dense cells, OOD on purpose).
- **NB02** — uses NB01's saved output or falls back to a synthetic ground-truth set.
- **NB03a** — `make_clean_image()` + `add_noise()` for a low-SNR pair with paired clean reference.
- **NB03b** — irregular blob structures designed to fall outside Cellpose's training distribution.
- **NB04** — each of the five mini-workflows (CARE, fnet, pix2pix, Deep-STORM, YOLOv2-style) generates 64-128 paired training samples inline.

### One canonical download with synthetic fallback (Notebook 00)

The setup self-check (NB00) tries to download a single small Cellpose example image (`http://www.cellpose.org/static/data/img02.png`). If the network is blocked, the notebook falls back to a synthetic 200x200 image with eight bright blobs.

### One large model download with simulated fallback (NB03b)

NB03b downloads the SAM ViT-B checkpoint (`sam_vit_b_01ec64.pth`, ~360 MB) on first run. If the download fails, the notebook drops to `SIMULATE_SAM=True` mode where SAM behavior is mocked with a simple circular mask around each prompt point — the workflow still demonstrates the prompt-based segmentation pattern.

The `.gitignore` excludes `sam_vit_*.pth` so this checkpoint is never tracked by Git.

### Live registry browse (NB04)

NB04's BioImage Model Zoo section calls `bioimageio.core` to query the BiMZ registry live and download a small pretrained model. If the registry is unreachable, the notebook falls back to a curated three-entry list so the cells still demonstrate the API.


## Why no bundled datasets

Two reasons:

1. **Reproducibility.** The notebooks generate or fetch their own data, so the labs survive any change to a dataset URL or hosting service. Earlier drafts of this workshop pointed at BBBC and externally-hosted training images — both are still useful references but neither is *required* for the labs to run.
2. **Distribution.** Bundling images in the repo would push the size from ~50 MB to several GB. Workshop attendees on free-tier Colab can fetch what they need on demand; nothing has to be pre-staged on a USB drive.


## How to use this notebook

1. **Pick the block** that matches where your data lives (local files, Drive, etc.).
2. **Copy the code** into the lab notebook you want to run, or run it here and switch back.
3. **Bind the working variables** the lab notebook expects (see the reference table at the bottom).
4. **Re-run the lab notebook** from the cell *after* its decision block.

The blocks are intentionally minimal — short, well-commented, paste-ready. None of them depend on each other.

## Block A — Workshop default (MABC samples + canonical fallback)

This is what each lab notebook does by default. Shown here for completeness — you don't normally run this block manually; just leave the lab notebook's decision dropdown set to `"MABC hosted"`.

In [ ]:
import os, urllib.request, urllib.error, tempfile
import numpy as np

NB_ID = "01_cellpose_segmentation"   # change to your target NB id
MABC_URL = f"https://microscopy-core-ismms.github.io/ImageAnalysisCourse/data/mabc/{NB_ID}.npz"

cache = os.path.join(tempfile.gettempdir(), os.path.basename(MABC_URL))
if not os.path.exists(cache):
    urllib.request.urlretrieve(MABC_URL, cache)

data = np.load(cache, allow_pickle=True)
real_imgs = list(data["images"])
real_filenames = list(data["filenames"]) if "filenames" in data.files else None
metadata = data["metadata"].item() if "metadata" in data.files else {}

print(f"Loaded {len(real_imgs)} images from MABC sample {NB_ID}.")
print(f"  shape per image: {real_imgs[0].shape}, dtype: {real_imgs[0].dtype}")
print(f"  source: {metadata.get('source', '(unknown)')}")


## Block B — Load your own local files

Use this when your images are on the same machine as the notebook. On Colab, "local" means the runtime VM — not your laptop. To work on your laptop's files in Colab, mount Drive (Block C) or upload them to the runtime first (`from google.colab import files; files.upload()`).

In [ ]:
from pathlib import Path
import numpy as np

# Edit these paths.
DATA_DIR = Path("/content/my_images")          # folder with your TIFF/PNG files
PATTERN = "*.tif"                                # glob pattern
N_LIMIT = 8                                      # how many to load

# Try tifffile first (handles multi-page, 16-bit), fall back to PIL.
try:
    import tifffile
    _read = lambda p: tifffile.imread(str(p))
except ImportError:
    from PIL import Image
    _read = lambda p: np.array(Image.open(p))

paths = sorted(DATA_DIR.glob(PATTERN))[:N_LIMIT]
real_imgs = [_read(p) for p in paths]
real_filenames = [p.name for p in paths]

print(f"Loaded {len(real_imgs)} files from {DATA_DIR}.")
for fn, im in zip(real_filenames, real_imgs):
    print(f"  {fn}: shape={im.shape}, dtype={im.dtype}")


**TIFF gotchas.** Multi-page TIFF (Z-stacks) returns a 3D array `(Z, H, W)`. Multi-channel TIFF can return `(C, H, W)` or `(H, W, C)` depending on how it was saved. The lab notebooks generally handle both, but if a downstream cell errors on shape, do `print(real_imgs[0].shape)` and slice/transpose to `(H, W)` for grayscale or `(H, W, 3)` for RGB.

**Other formats** — for proprietary microscopy formats (`.lif`, `.nd2`, `.czi`, `.lsm`):

```python
%pip install bioio bioio-lif bioio-nd2  # one of: -lif, -nd2, -czi, -lsm
from bioio import BioImage
img = BioImage("/content/my_image.lif")
real_imgs = [img.get_image_data("YX", T=0, C=0, Z=z) for z in range(img.dims.Z)]
```

## Block C — Mount Google Drive (Colab only)

Use this when your images live in your Google Drive. Authentication runs once per Colab runtime; subsequent cells access files at paths like `/content/drive/MyDrive/...`.

In [ ]:
# Colab-only: mount Google Drive at /content/drive/
from google.colab import drive
drive.mount("/content/drive")

# Now your Drive files are at /content/drive/MyDrive/<your-folder>/...
# Combine with Block B above to load:
from pathlib import Path
DATA_DIR = Path("/content/drive/MyDrive/workshop_images")
PATTERN = "*.tif"
# ... continue with Block B's loading code


**First-time auth.** The cell above pops up a Google sign-in flow. Sign in with the same Google account that owns the Drive folder. Authorization persists for the runtime session (~12 hours).

**Path tip.** If your folder is in a Shared Drive (org account), the path is `/content/drive/Shareddrives/<DriveName>/...` — note the lowercase `d` in `drive` and the capital `Shared`.

## Block D — Pull from any public URL

Use this when your data lives on a public web URL (your lab's website, Zenodo, Figshare, GitHub raw, etc.) — anything reachable via HTTPS without authentication.

In [ ]:
import os, urllib.request, tempfile, zipfile
from pathlib import Path
import numpy as np

# Edit these.
URL = "https://example.org/path/to/your-data.zip"
N_LIMIT = 8

cache = os.path.join(tempfile.gettempdir(), os.path.basename(URL))
if not os.path.exists(cache):
    print(f"Downloading {URL}...")
    urllib.request.urlretrieve(URL, cache)

# If it's a zip, extract; otherwise just load.
if cache.endswith(".zip"):
    extract_dir = cache + "_extracted"
    if not os.path.isdir(extract_dir):
        os.makedirs(extract_dir, exist_ok=True)
        with zipfile.ZipFile(cache) as zf:
            zf.extractall(extract_dir)
    img_paths = sorted(Path(extract_dir).rglob("*.tif"))[:N_LIMIT]
else:
    img_paths = [Path(cache)]

try:
    import tifffile
    _read = lambda p: tifffile.imread(str(p))
except ImportError:
    from PIL import Image
    _read = lambda p: np.array(Image.open(p))

real_imgs = [_read(p) for p in img_paths]
real_filenames = [p.name for p in img_paths]
print(f"Loaded {len(real_imgs)} images from {URL}.")


**Citation hygiene.** If the URL points at a published dataset (BBBC, GigaDB, BIA, IDR, etc.), capture the dataset name + DOI/URL in your notebook so anyone reading it can trace the data. The workshop's `datasets_audit.md` covers the most common canonical sources.

## Block E — HuggingFace Datasets

Use this when your dataset is hosted on the [HuggingFace Hub](https://huggingface.co/datasets). Many recent bioimage benchmarks (STimage-1K4M, BIN-1, etc.) live there. HuggingFace caches downloads in `~/.cache/huggingface/` so subsequent runs are fast.

In [ ]:
%pip install --quiet datasets
from datasets import load_dataset
import numpy as np

# Edit this.
DATASET_ID = "your-org/your-dataset"   # e.g., "polinaeterna/cifar10"
SPLIT = "train"
N_LIMIT = 8

ds = load_dataset(DATASET_ID, split=f"{SPLIT}[:{N_LIMIT}]")
# HF Datasets returns dicts; the image key is usually "image" or "img"
image_key = next((k for k in ds.column_names if k.lower() in ("image", "img", "pixel_values")), None)
if image_key is None:
    raise KeyError(f"No image column found. Available columns: {ds.column_names}")

real_imgs = [np.array(row[image_key]) for row in ds]
real_filenames = [f"{DATASET_ID}#{i}" for i in range(len(real_imgs))]
print(f"Loaded {len(real_imgs)} images from {DATASET_ID}.")


**Gated datasets.** Some HF datasets require accepting a license on the website first, plus a `huggingface-cli login` token. The `load_dataset` call will raise a clear error with a link if so.

## Block F — Mt Sinai Box / OneDrive (institutional storage)

**Short version: you can't fetch directly from Mt Sinai Box or institutional OneDrive in a Colab notebook**, because both require SSO authentication that doesn't work cleanly in a Jupyter cell.

**Workaround.** The cleanest path is to mirror the specific files you need into a personal Google Drive folder, then use Block C to mount that. From the Mt Sinai end:

1. Open the file in Box / OneDrive on your Mac.
2. Right-click → Download (or open in the desktop client and copy out).
3. Drop the file into your `MyDrive/workshop_images/` folder.
4. In Colab, mount Drive (Block C above) and load.

If you need to pull a Mt Sinai Box file at scale (>1 GB or many files), the Box API is the right answer — but configuring Box's OAuth flow inside a workshop slot is friction. **Recommend doing the data prep step on your laptop, not in Colab.**

For purely public Mt Sinai resources (anything published on a Mt Sinai-affiliated GitHub or website), Block D (public URL) works fine.

## Block G — Run notebooks locally instead of on Colab

Colab is the workshop default — zero-install, free GPU, identical environment for every attendee. But you can also run the lab notebooks on your own machine.

### Why local?
- **Bigger data.** Colab's free tier disconnects after ~12 hours and has limited disk. For multi-GB datasets, local is more practical.
- **Faster iteration.** No re-installing packages every fresh runtime.
- **Privacy.** If your data isn't public, mounting Drive may be inappropriate.

### Setup

```bash
git clone https://github.com/microscopy-Core-ISMMS/ImageAnalysisCourse.git
cd ImageAnalysisCourse

# Conda is recommended:
conda env create -f env/environment.yml
conda activate ai-microscopy-2026

# Or pip (less reliable for some packages — micro-sam, cellpose can be tricky):
pip install -r env/requirements_notebooks.txt

# Launch JupyterLab and open the notebook you want:
jupyter lab notebooks/01_cellpose_segmentation.ipynb
```

### GPU notes
- **CUDA on Linux/Windows.** Most labs benefit from a GPU. Cellpose, SAM, and the larger U-Nets in NB07/12 will run on CPU but slowly (5–30× longer).
- **Apple Silicon Mac.** PyTorch's `mps` backend works for most operations. Set `device = "mps"` and expect ~50–80% of CUDA performance for the small models in this workshop.
- **No GPU.** Everything still runs; expect total runtime around 60–90 min for the labs that train models.

### Where to put data
Put your input images anywhere you like and edit the `DATA_DIR` constant in the lab notebook's loading cell (or in Block B above) to point at it. The lab notebooks don't enforce a specific layout — they just consume `real_imgs` as a list of NumPy arrays.

## Variable name reference

After you load your data into `real_imgs` (and optionally `real_filenames`, `real_metadata`), each lab notebook expects specific working variables to be bound. The lab notebook's decision block does this binding automatically when MABC / canonical succeeds. If you load data manually here, you'll need to do the binding yourself.

| NB | Working variables expected | Suggested binding |
|---|---|---|
| **NB01** Cellpose pretrained | `img_easy`, `img_hard`, `img_easy_synth`, `img_hard_synth`, `img` | `img_easy = real_imgs[0]; img_hard = real_imgs[1]; img_easy_synth = real_imgs[0]; img_hard_synth = real_imgs[1]; img = real_imgs[0]` |
| **NB02** Validation/QC | `gt_easy`, `gt_hard` (binary masks) | `gt_easy = (real_imgs[0] > 0).astype(np.uint8); gt_hard = (real_imgs[-1] > 0).astype(np.uint8)` |
| **NB03a** Denoising | `clean`, `noisy` | `clean = real_imgs[0].astype(float)/255; noisy = clean + 0.10*np.random.randn(*clean.shape)` |
| **NB03b** Foundation seg (SAM) | `img` | `img = real_imgs[0].astype(float)/255` |
| **NB06** Virtual staining | `X_train`, `Y_train`, `X_test`, `Y_test` | Split `real_imgs` 6/2 train/test, pair channels appropriately |
| **NB07** Super-resolution | `hr_train`, `hr_test`, `lr_train_small`, `lr_train`, `lr_test_small`, `lr_test`, `n_train`, `n_test` | Split 6/2; LR via Gaussian-blur + 0.5× downsample + bicubic-back-up |
| **NB09** Cellpose finetune | `train_images`, `train_labels`, `test_images`, `test_labels`, `img` | `train_images = real_imgs[:6]; test_images = real_imgs[6:]; train_labels/test_labels = zero masks unless you have GT` |
| **NB12** Deconvolution | `X_train_clean`, `X_train_blurred`, `X_train_blurred_noisy`, `X_test_*`, `n_train`, `n_test` | Same 6/2 split; blur via Gaussian σ≈2, noise via N(0, 0.05) |
| **NB13** Validation case study | `img`, `TEST_IMAGES` | `img = real_imgs[0]; TEST_IMAGES = {f'real_{i}': (real_imgs[i], None) for i in range(min(4, len(real_imgs)))}` |
| **NB14** Spot detection | `train_images`, `train_centers`, `test_images`, `test_centers` | Split 6/2; centers = `[None] * N` if no GT spot coords |
| **NB16** WSI → transcriptomics | `he_tiles` (list of RGB arrays) | `he_tiles = [np.asarray(img) for img in real_imgs]` |

The decision-block code in each lab notebook embeds the right binding logic, so most attendees never touch this table. It's here for the case where you load data manually and need to know what to assign.

## After the workshop — bringing your own data

The notebooks are written so swapping in your own image typically requires changing **one path** at the top of the relevant notebook. Look for the cells that load `data.cell()` or call `make_easy_image()` and replace with `tifffile.imread("path/to/your_image.tif")`. The downstream cells are mostly format-agnostic (numpy arrays in, numpy arrays out).

For larger projects that need real benchmark datasets:

- **[BBBC — Broad Bioimage Benchmark Collection](https://bbbc.broadinstitute.org/)** — curated benchmark microscopy datasets with documented ground truth. Good for validation work beyond NB02.
- **[BioImage Archive (EMBL-EBI)](https://www.ebi.ac.uk/bioimage-archive/)** — public archive for biological image data.
- **[IDR — Image Data Resource](https://idr.openmicroscopy.org)** — public reference imaging datasets.
- **[Cellpose example data](https://www.cellpose.org/static/data/)** — Cellpose-hosted samples used in the original publications.
- **[Allen Cell Image Library](http://www.cellimagelibrary.org)** — curated cell biology images.

These are listed in full on the [Resources page](../resources) with citations.


## Licensing

- **scikit-image bundled images** — distributed under scikit-image's BSD 3-Clause license; reusable for educational and research purposes with attribution.
- **Synthetic generators** — produced inline by code in this repository; covered by this repository's MIT License (for the code) and CC-BY-4.0 (for the documentation around it).
- **Cellpose example image** — courtesy of the Cellpose authors; verify their site for current licensing if you redistribute.
- **SAM checkpoint** — released under the SAM authors' license; see https://github.com/facebookresearch/segment-anything.
- **MABC sample npz files (`data/mabc/*.npz`)** — Microscopy and Advanced Bioimaging Core, Mt Sinai. Workshop sample 2026. CC-BY 4.0.

Per-notebook citations and attributions are in [`acknowledgments.md`](../acknowledgments) at the repo root.


## Convention — Z-selection and Z-projection widgets for volumes

Any lab notebook in this book that consumes a Z-stack (cells3d, PAM mice, BBBC, custom) exposes a small form widget at the top of the canonical-data section. This page documents the convention so future notebooks can lift it verbatim.

### Standard widget (most notebooks)

```python
# @title Volume slice selection (canonical tier — cells3d) { run: "auto", display-mode: "form" }
Z_MIN  = 20  # @param {type:"slider", min:0, max:59, step:1}
Z_MAX  = 45  # @param {type:"slider", min:0, max:59, step:1}
Z_STEP = 1   # @param {type:"slider", min:1, max:10, step:1}
Z_PROJECTION = "none (use all slices)"  # @param ["none (use all slices)", "max intensity", "mean", "median", "single mid-slice"]
```

### Standard helper

```python
def apply_z_selection(volume_zyx, z_min=None, z_max=None, z_step=None, projection=None):
    """Return (array, label). For projections, returns 2D; else returns 3D sub-stack."""
    import numpy as _np
    if z_min is None: z_min = Z_MIN
    if z_max is None: z_max = Z_MAX
    if z_step is None: z_step = Z_STEP
    if projection is None: projection = Z_PROJECTION
    sub = volume_zyx[z_min:z_max:z_step]
    if projection == "max intensity":     return sub.max(axis=0),     f"max-proj Z={z_min}..{z_max}:{z_step}"
    if projection == "mean":              return sub.mean(axis=0),    f"mean-proj Z={z_min}..{z_max}:{z_step}"
    if projection == "median":            return _np.median(sub, axis=0), f"median-proj Z={z_min}..{z_max}:{z_step}"
    if projection == "single mid-slice":  mid = (z_min + z_max) // 2; return volume_zyx[mid], f"single Z={mid}"
    return sub, f"all Z={z_min}..{z_max}:{z_step} ({len(sub)} slices)"
```

### What each option does

| Option | Output shape | Use when |
|--------|--------------|----------|
| `none (use all slices)` | 3D sub-stack | Per-slice training (NB06 default), volume processing |
| `max intensity` | 2D | Sparse bright signal (FISH spots, single molecules); dominant brightness wins |
| `mean` | 2D | Smooths noise; baseline "what's there on average" |
| `median` | 2D | Robust to outliers / hot pixels / dust |
| `single mid-slice` | 2D | One representative slice at `(Z_MIN+Z_MAX)//2`; ignores `Z_STEP` |

### Variant: NB10 (`BASELINE_*`)

NB10's main flow uses the full 3D volume by design. Its widget is named `BASELINE_Z_MIN` / `BASELINE_Z_MAX` / `BASELINE_Z_STEP` / `BASELINE_PROJECTION` to make clear the widget controls a **2D-projection baseline** that runs alongside the 3D pipeline, not a slice subset for the main path. The default `BASELINE_PROJECTION = "max intensity"` produces an actual 2D baseline image rather than skipping the baseline.

### Variant: NB03a (axial averaging)

NB03a doesn't use `apply_z_selection`. It has a single-slice `Z_INDEX` slider and computes an **axial-averaged clean reference** from adjacent slices — a denoising-specific operation that's narrower in scope than the general projection options here. Don't merge these.


## Further resources

- **Workshop dataset audit:** [`datasets_audit.md`](https://github.com/microscopy-Core-ISMMS/ImageAnalysisCourse/blob/2026-workshop/datasets_audit.md) — license + format notes for every canonical source the workshop touches (BBBC, BIA, IDR, Allen Cell, BSCCM, GigaDB, CIL, Cellpose, NIST NexusLIMS).
- **MABC manifest:** [`data/MANIFEST.json`](https://github.com/microscopy-Core-ISMMS/ImageAnalysisCourse/blob/2026-workshop/data/MANIFEST.json) — sha256 + size of each `data/mabc/<nb>.npz`.
- **Bake script:** [`scripts/bake_dataset_samples.py`](https://github.com/microscopy-Core-ISMMS/ImageAnalysisCourse/blob/2026-workshop/scripts/bake_dataset_samples.py) — produces the MABC npz files from raw inputs.

---

*This notebook is part of the [AI for Microscopy Image Analysis](https://microscopy-core-ismms.github.io/ImageAnalysisCourse/) workshop.*